# Distance-Specific GraphSAGE Training

This notebook is a **training worker** that trains separate **specialist** GraphSAGE models for each specified code distance.

**Usage:**
1. Set the `DISTANCES` parameter in the Configuration cell (e.g., `[3, 5, 7]`)
2. Run all cells
3. A separate model is trained for **each** distance in the list
4. Training artifacts are saved to `models/`, `results/`, and `plots/`

**Features:**
- Trains one specialist model per distance (e.g., `[3, 5, 7]` → 3 separate models)
- Uses best hyperparameters from tuning experiments
- Trains for 50 epochs with comprehensive logging
- **Checkpoint resumption**: Saves progress after every epoch; automatically resumes if interrupted
- Saves model checkpoint, training metrics JSON, plots, and CSV summary per distance
- Supports both local and Google Colab execution

**Checkpoint Behavior:**
- Checkpoints are saved to `models/d{distance}.pt` after each epoch
- If training crashes, simply re-run to continue from the last completed epoch
- Completed training is marked; re-running will skip training unless the checkpoint is deleted

## 1. Configuration (USER INPUT)

In [6]:
# =============================================================================
# USER CONFIGURATION - MODIFY THIS CELL
# =============================================================================

# Specify which distances to train SEPARATE specialist models for
# Each distance in this list gets its own dedicated model
# Examples:
#   DISTANCES = [3]           # Train one model for d=3
#   DISTANCES = [3, 5, 7]     # Train 3 separate models: d=3, d=5, d=7
#   DISTANCES = [5, 7, 9, 11] # Train 4 separate models

DISTANCES = [3, 5, 7, 9, 11]  # <-- MODIFY THIS

print(f"Configuration:")
print(f"  Distances to train: {DISTANCES}")
print(f"  Number of models to train: {len(DISTANCES)}")
for d in DISTANCES:
    print(f"    - d{d} specialist model")

Configuration:
  Distances to train: [3, 5, 7, 9, 11]
  Number of models to train: 5
    - d3 specialist model
    - d5 specialist model
    - d7 specialist model
    - d9 specialist model
    - d11 specialist model


## 2. Imports and Setup

In [1]:
# Install dependencies (uncomment if needed)
!pip install stim pymatching numpy matplotlib torch tqdm networkx pandas
!pip install torch_geometric

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
import json
import random
import time
import platform
from pathlib import Path
from datetime import datetime

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Research/QEC/quantum-error-correction/quantum-error-correction/code')
else:
    BASE_PATH = Path('../..')  # code/gSAGE/distances -> code/

sys.path.insert(0, str(BASE_PATH))

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import tqdm
from torch_geometric.loader import DataLoader

# Import from models.py
from models import (
    SurfaceCodeSampler,
    SparseGraph,
    GraphSAGEModel,
    GraphSAGE,
    DatasetCache,
)

# Set up paths for this experiment
DISTANCES_DIR = BASE_PATH / "gSAGE" / "distances"
MODELS_DIR = DISTANCES_DIR / "models" / "revised_training"
RESULTS_DIR = DISTANCES_DIR / "results" / "revised_training"
PLOTS_DIR = DISTANCES_DIR / "plots" / "revised_training"

# Create output directories
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")
if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    print(f"GPU: {GPU_NAME}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    GPU_NAME = None

print(f"\nPaths:")
print(f"  BASE_PATH: {BASE_PATH}")
print(f"  MODELS_DIR: {MODELS_DIR}")
print(f"  RESULTS_DIR: {RESULTS_DIR}")
print(f"  PLOTS_DIR: {PLOTS_DIR}")

C:\Users\Bill\AppData\Roaming\Python\Python313\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
C:\Users\Bill\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Using device: cuda
GPU: NVIDIA GeForce RTX 5070 Ti
CUDA version: 13.0

Paths:
  BASE_PATH: ..\..
  MODELS_DIR: ..\..\gSAGE\distances\models\revised_training
  RESULTS_DIR: ..\..\gSAGE\distances\results\revised_training
  PLOTS_DIR: ..\..\gSAGE\distances\plots\revised_training


## 3. Load Best Hyperparameters

In [11]:
# =============================================================================
# LOAD BEST HYPERPARAMETERS FROM TUNING
# =============================================================================

BEST_CONFIG_PATH = BASE_PATH / "gSAGE" / "tuning" / "best_model_config.json"

with open(BEST_CONFIG_PATH, 'r') as f:
    best_config = json.load(f)

# Extract hyperparameters
BEST_HYPERPARAMS = {
    'num_layers': best_config['model_params']['num_layers'],
    'hidden_dim': best_config['model_params']['hidden_dim'],
    'learning_rate': best_config['training_params']['learning_rate'],
    'dropout': best_config['model_params']['dropout'],
    'aggr': best_config['model_params']['aggr'],
}

print(f"Loaded best hyperparameters from: {BEST_CONFIG_PATH}")
print(f"\nTuning results:")
print(f"  Config ID: {best_config['config_id']}")
print(f"  Validation accuracy: {best_config['performance']['val_accuracy']*100:.2f}%")
print(f"  Test accuracy: {best_config['performance']['test_accuracy']*100:.2f}%")
print(f"\nHyperparameters:")
for k, v in BEST_HYPERPARAMS.items():
    print(f"  {k}: {v}")

Loaded best hyperparameters from: ..\..\gSAGE\tuning\best_model_config.json

Tuning results:
  Config ID: 17
  Validation accuracy: 90.22%
  Test accuracy: 90.28%

Hyperparameters:
  num_layers: 5
  hidden_dim: 128
  learning_rate: 0.0005
  dropout: 0.0
  aggr: max


## 4. Training Configuration

In [10]:
# =============================================================================
# TRAINING CONFIGURATION (applies to each specialist model)
# =============================================================================

# Training parameters
EPOCHS = 50
BATCH_SIZE = 256

# Refresh/eval cadence
REFRESH_EVERY = 10          # regenerate TRAINING graphs every N epochs
VALIDATE_EVERY = 10         # run validation every N epochs (at refresh boundaries)

# Dataset parameters (per specialist model)
SAMPLES_PER_MODEL = 1_000_000  # Samples for each specialist model

# Training samples per refresh block (per model)
TRAIN_SAMPLES_PER_BLOCK = int(SAMPLES_PER_MODEL * 0.8)

# Error rate distribution
P_VALUES = [0.001, 0.003, 0.005, 0.007]
P_WEIGHTS = [0.25, 0.25, 0.25, 0.25]

# Graph construction
K_NEIGHBORS = 6
IN_CHANNELS = 5  # Node features: [X?, Z?, d_North, d_West, d_time]

# Reproducibility
SEED = 42

# Dataset split ratios (used for cached eval split)
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

print(f"Training Configuration (per specialist model):")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Refresh every: {REFRESH_EVERY} epochs")
print(f"  Validate every: {VALIDATE_EVERY} epochs")
print(f"  Learning rate: {BEST_HYPERPARAMS['learning_rate']}")
print(f"\nDataset Configuration (per model):")
print(f"  Samples per model: {SAMPLES_PER_MODEL:,}")
print(f"  Training samples per refresh block: {TRAIN_SAMPLES_PER_BLOCK:,}")
print(f"  P values: {P_VALUES}")
print(f"  P weights: {P_WEIGHTS}")
print(f"\nModels to train: {len(DISTANCES)}")
for d in DISTANCES:
    print(f"  - d{d}: {SAMPLES_PER_MODEL:,} samples")
print(f"\nEval split ratios:")
print(f"  Train: {TRAIN_RATIO*100:.0f}%")
print(f"  Validation: {VAL_RATIO*100:.0f}%")
print(f"  Test: {TEST_RATIO*100:.0f}%")

Training Configuration (per specialist model):
  Epochs: 50
  Batch size: 256
  Refresh every: 10 epochs
  Validate every: 10 epochs


NameError: name 'BEST_HYPERPARAMS' is not defined

## 5. Dataset Generation/Loading

In [ ]:
def generate_or_load_dataset(d: int, n_samples: int, cache_name: str = None) -> list:
    """
    Generate or load a dataset for a specific distance.
    
    Args:
        d: Code distance
        n_samples: Number of samples to generate/load
        cache_name: Optional cache name to load from/save to
        
    Returns:
        List of PyG Data objects
    """
    cache = DatasetCache(base_path=BASE_PATH, device=device)
    
    # Try to load from cache first
    if cache_name:
        try:
            cache.load(cache_name, verbose=True)
            if cache.size() >= n_samples:
                print(f"  Loaded {n_samples:,} samples from cache '{cache_name}'")
                return cache.get_graphs(n=n_samples, shuffle=True)
            else:
                print(f"  Cache has {cache.size():,} samples, need {n_samples:,}. Growing...")
                cache.ensure_size(n_samples, verbose=True)
                cache.save(cache_name)
                return cache.get_graphs(n=n_samples, shuffle=True)
        except FileNotFoundError:
            print(f"  Cache '{cache_name}' not found. Generating new dataset...")
    
    # Generate new dataset
    cache.generate(
        d=d,
        n_samples=n_samples,
        p_values=P_VALUES,
        p_weights=P_WEIGHTS,
        k_neighbors=K_NEIGHBORS,
        verbose=True
    )
    
    # Save to cache if name provided
    if cache_name:
        cache.save(cache_name)
    
    return cache.get_graphs(shuffle=True)


def _mix_seed(base: int, scope: str, block_idx: int) -> int:
    """Deterministic seed mixer (uint32-ish)."""
    h = base
    for ch in str(scope):
        h = (h * 1000003 + ord(ch)) % (2**32 - 1)
    h = (h * 1009 + block_idx * 9176) % (2**32 - 1)
    return int(h)


def generate_fresh_training_graphs_single(d: int, n_samples: int, block_idx: int) -> list:
    """Generate a fresh TRAINING graph dataset for a single distance (no disk cache).
    
    Args:
        d: Code distance
        n_samples: Number of samples to generate
        block_idx: Block index for deterministic seeding
        
    Returns:
        List of PyG Data objects
    """
    print(f"\n[train refresh] Generating d={d} training graphs for block {block_idx} (n={n_samples:,})")

    # Deterministic seeding for reproducibility
    seed_i = _mix_seed(SEED, f"train_d{d}", block_idx)
    random.seed(seed_i)
    np.random.seed(seed_i % (2**32 - 1))
    torch.manual_seed(seed_i)

    # Generate fresh training data (no disk cache)
    cache = DatasetCache(base_path=BASE_PATH, device=device)
    cache.generate(
        d=d,
        n_samples=n_samples,
        p_values=P_VALUES,
        p_weights=P_WEIGHTS,
        k_neighbors=K_NEIGHBORS,
        verbose=True,
    )
    graphs = cache.get_graphs(shuffle=True)
    del cache

    print(f"Training graphs ready: {len(graphs):,} samples")
    return graphs

In [ ]:
def load_single_distance_dataset(d: int, n_samples: int) -> tuple:
    """
    Load dataset for a single distance.
    
    Args:
        d: Code distance
        n_samples: Number of samples to load
        
    Returns:
        Tuple of (graphs, dataset_info)
    """
    print(f"\n{'='*60}")
    print(f"Loading dataset for d={d}")
    print(f"Samples: {n_samples:,}")
    print(f"{'='*60}")
    
    cache_name = f"d{d}_baseline"
    print(f"\n[d={d}] Loading {n_samples:,} samples...")
    
    start_time = time.time()
    graphs = generate_or_load_dataset(d, n_samples, cache_name)
    load_time = time.time() - start_time
    
    dataset_info = {
        f'd{d}': {
            'samples': len(graphs),
            'load_time_seconds': load_time
        }
    }
    print(f"  Loaded {len(graphs):,} samples (took {load_time:.1f}s)")
    
    # Shuffle dataset
    print(f"\nShuffling dataset...")
    random.seed(SEED)
    random.shuffle(graphs)
    
    print(f"\nDataset ready: {len(graphs):,} total samples")
    return graphs, dataset_info

In [ ]:
def split_dataset(graphs: list, train_ratio: float, val_ratio: float, seed: int):
    """
    Split dataset into train/val/test sets.
    
    Args:
        graphs: List of PyG Data objects
        train_ratio: Proportion for training
        val_ratio: Proportion for validation
        seed: Random seed
        
    Returns:
        Tuple of (train_graphs, val_graphs, test_graphs)
    """
    random.seed(seed)
    indices = list(range(len(graphs)))
    random.shuffle(indices)
    
    n = len(graphs)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    
    train_idx = indices[:train_end]
    val_idx = indices[train_end:val_end]
    test_idx = indices[val_end:]
    
    train_graphs = [graphs[i] for i in train_idx]
    val_graphs = [graphs[i] for i in val_idx]
    test_graphs = [graphs[i] for i in test_idx]
    
    print(f"\nDataset split:")
    print(f"  Train: {len(train_graphs):,} samples ({len(train_graphs)/n*100:.1f}%)")
    print(f"  Validation: {len(val_graphs):,} samples ({len(val_graphs)/n*100:.1f}%)")
    print(f"  Test: {len(test_graphs):,} samples ({len(test_graphs)/n*100:.1f}%)")
    
    return train_graphs, val_graphs, test_graphs

In [ ]:
# -----------------------------------------------------------------------------
# Data loading happens per-distance in the main training loop below.
# Each specialist model gets its own val/test split from its distance's baseline cache.
# Training data is refreshed every REFRESH_EVERY epochs (not cached).
# -----------------------------------------------------------------------------
print("Data loading functions defined. Actual loading happens per-distance in training loop.")


In [ ]:
# =============================================================================
# CHECKPOINT FUNCTIONS
# =============================================================================

def load_checkpoint(experiment_name: str, models_dir: Path) -> dict:
    """
    Load an existing checkpoint if available.
    
    Args:
        experiment_name: Name of the experiment (e.g., 'd3_d11')
        models_dir: Directory where models are saved
        
    Returns:
        Dict with checkpoint data if found, None otherwise.
        Checkpoint contains: current_epoch, state_dict, optimizer_state_dict, 
                            epoch_metrics, training_complete, and config info.
    """
    checkpoint_path = models_dir / f"{experiment_name}.pt"
    
    if not checkpoint_path.exists():
        print(f"No checkpoint found at: {checkpoint_path}")
        return None
    
    print(f"Loading checkpoint from: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    
    # Check if this is a checkpoint (has current_epoch) vs a final model
    if 'current_epoch' not in checkpoint:
        print(f"  Found final model (no epoch tracking). Starting fresh training.")
        return None
    
    current_epoch = checkpoint['current_epoch']
    training_complete = checkpoint.get('training_complete', False)
    
    if training_complete:
        print(f"  Training already complete ({current_epoch} epochs). Skipping.")
        return checkpoint
    
    print(f"  Resuming from epoch {current_epoch + 1}")
    print(f"  Last train accuracy: {checkpoint['epoch_metrics']['train_accuracy'][-1]*100:.2f}%")
    print(f"  Last val accuracy: {checkpoint['epoch_metrics']['val_accuracy'][-1]*100:.2f}%")
    
    return checkpoint


def save_checkpoint(
    model,
    optimizer,
    epoch: int,
    epoch_metrics: dict,
    experiment_name: str,
    distances: list,
    hyperparams: dict,
    training_config: dict,
    models_dir: Path,
    training_complete: bool = False
):
    """
    Save a training checkpoint.
    
    Args:
        model: GraphSAGE model wrapper
        optimizer: Optimizer with current state
        epoch: Current completed epoch (1-indexed)
        epoch_metrics: Dict of accumulated metrics
        experiment_name: Name of the experiment
        distances: List of code distances
        hyperparams: Model hyperparameters
        training_config: Training configuration dict
        models_dir: Directory to save checkpoint
        training_complete: Whether training is finished
    """
    checkpoint_path = models_dir / f"{experiment_name}.pt"
    
    checkpoint = {
        'state_dict': model.model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'current_epoch': epoch,
        'epoch_metrics': epoch_metrics,
        'training_complete': training_complete,
        'config': model._config,
        'experiment_name': experiment_name,
        'distances': distances,
        'hyperparams': hyperparams,
        'training_config': training_config,
        'timestamp': datetime.now().isoformat(),
    }
    
    torch.save(checkpoint, checkpoint_path)
    
    if training_complete:
        print(f"Final checkpoint saved to: {checkpoint_path}")
    # Don't print for every epoch to avoid clutter


print("Checkpoint functions defined.")

## 6. Training with Heavy Logging

In [ ]:
def evaluate_model(model, graphs, device, batch_size=256):
    """
    Evaluate model accuracy on a set of graphs.
    
    Args:
        model: GraphSAGE model wrapper
        graphs: List of PyG Data objects
        device: Torch device
        batch_size: Batch size for evaluation
        
    Returns:
        Accuracy as a float
    """
    model.model.eval()
    loader = DataLoader(graphs, batch_size=batch_size, shuffle=False)
    
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = model.model(batch)
            y = batch.y.float().view(-1, 1)
            correct += ((pred > 0.5).float() == y).sum().item()
            total += y.size(0)
    
    return correct / total if total > 0 else 0.0

In [ ]:
def train_with_logging(
    model,
    val_graphs,
    epochs,
    batch_size,
    lr,
    device,
    # Training data refresh
    generate_train_graphs_fn,
    refresh_every: int = 10,
    validate_every: int = 10,
    # Checkpoint parameters
    experiment_name: str = None,
    distances: list = None,
    hyperparams: dict = None,
    training_config: dict = None,
    models_dir: Path = None,
    # Resumption parameters
    start_epoch: int = 0,
    initial_metrics: dict = None,
    optimizer_state_dict: dict = None,
    verbose: bool = True
):
    """Train model with per-epoch logging, checkpointing, and training-data refresh.

    - Training graphs are regenerated every `refresh_every` epochs via `generate_train_graphs_fn(block_idx)`.
    - Validation is evaluated every `validate_every` epochs and at the final epoch.
    - Optimizer state is preserved across refresh blocks.
    """
    import gc

    model.model.train()
    optimizer = torch.optim.Adam(model.model.parameters(), lr=lr)
    loss_fn = torch.nn.BCELoss()

    # Restore optimizer state if resuming
    if optimizer_state_dict is not None:
        optimizer.load_state_dict(optimizer_state_dict)
        if verbose:
            print("Restored optimizer state from checkpoint")

    # Initialize or restore logging containers
    if initial_metrics is not None:
        epoch_metrics = initial_metrics.copy()
        if verbose:
            print(f"Restored {len(epoch_metrics['epoch'])} epochs of metrics from checkpoint")
    else:
        epoch_metrics = {
            'epoch': [],
            'train_loss': [],
            'train_accuracy': [],
            'val_accuracy': [],
            'epoch_time_seconds': [],
            'learning_rate': [],
            'gpu_memory_mb': [],
            'train_block_idx': [],
            'train_samples': [],
        }

    checkpoint_enabled = all([
        experiment_name is not None,
        distances is not None,
        hyperparams is not None,
        training_config is not None,
        models_dir is not None
    ])

    if verbose:
        print(f"\n{'='*70}")
        print(f"Training: {model.nickname}")
        print(f"{'='*70}")
        print(f"Epochs: {start_epoch + 1} to {epochs} | Batch size: {batch_size} | LR: {lr}")
        print(f"Validation samples: {len(val_graphs):,}")
        print(f"Refresh every: {refresh_every} epochs | Validate every: {validate_every} epochs")
        print(f"Checkpointing: {'Enabled' if checkpoint_enabled else 'Disabled'}")
        print(f"{'='*70}\n")

    total_start_time = time.time()

    current_block_idx = start_epoch // refresh_every
    train_graphs = generate_train_graphs_fn(current_block_idx)
    loader = DataLoader(
        train_graphs,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
    )

    for epoch in range(start_epoch, epochs):
        epoch_start_time = time.time()

        desired_block_idx = epoch // refresh_every
        if desired_block_idx != current_block_idx:
            current_block_idx = desired_block_idx
            del train_graphs
            gc.collect()
            train_graphs = generate_train_graphs_fn(current_block_idx)
            loader = DataLoader(
                train_graphs,
                batch_size=batch_size,
                shuffle=True,
                num_workers=0,
                pin_memory=True,
            )

        model.model.train()
        epoch_loss = 0.0
        epoch_correct = 0
        epoch_total = 0

        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}", disable=not verbose, leave=False)

        for batch_data in pbar:
            batch_data = batch_data.to(device)
            pred = model.model(batch_data)
            y = batch_data.y.float().view(-1, 1)

            loss = loss_fn(pred, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            epoch_correct += ((pred > 0.5).float() == y).sum().item()
            epoch_total += y.size(0)

            current_acc = 100.0 * epoch_correct / epoch_total
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{current_acc:.1f}%'})

        pbar.close()

        avg_train_loss = epoch_loss / len(loader)
        train_accuracy = epoch_correct / epoch_total

        do_validate = ((epoch + 1) % validate_every == 0) or (epoch + 1 == epochs)
        if do_validate:
            val_accuracy = evaluate_model(model, val_graphs, device, batch_size)
        else:
            val_accuracy = float('nan')

        epoch_time = time.time() - epoch_start_time

        if torch.cuda.is_available():
            gpu_memory_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)
            torch.cuda.reset_peak_memory_stats()
        else:
            gpu_memory_mb = 0.0

        epoch_metrics['epoch'].append(epoch + 1)
        epoch_metrics['train_loss'].append(avg_train_loss)
        epoch_metrics['train_accuracy'].append(train_accuracy)
        epoch_metrics['val_accuracy'].append(val_accuracy)
        epoch_metrics['epoch_time_seconds'].append(epoch_time)
        epoch_metrics['learning_rate'].append(lr)
        epoch_metrics['gpu_memory_mb'].append(gpu_memory_mb)
        epoch_metrics['train_block_idx'].append(current_block_idx)
        epoch_metrics['train_samples'].append(len(train_graphs))

        if verbose:
            val_str = f"{val_accuracy*100:.2f}%" if (val_accuracy == val_accuracy) else "(skipped)"
            print(f"Epoch {epoch+1:3d}/{epochs} | "
                  f"Loss: {avg_train_loss:.4f} | "
                  f"Train Acc: {train_accuracy*100:.2f}% | "
                  f"Val Acc: {val_str} | "
                  f"Block: {current_block_idx} | "
                  f"Time: {epoch_time:.1f}s | "
                  f"GPU: {gpu_memory_mb:.0f}MB")

        if checkpoint_enabled:
            is_final = (epoch + 1 == epochs)
            save_checkpoint(
                model=model,
                optimizer=optimizer,
                epoch=epoch + 1,
                epoch_metrics=epoch_metrics,
                experiment_name=experiment_name,
                distances=distances,
                hyperparams=hyperparams,
                training_config=training_config,
                models_dir=models_dir,
                training_complete=is_final
            )
            if verbose and is_final:
                print("  [Checkpoint saved - training complete]")

    total_time = time.time() - total_start_time

    if verbose:
        print(f"\n{'='*70}")
        print("Training complete!")
        print(f"Total time: {total_time/60:.1f} minutes")
        print(f"Final train loss: {epoch_metrics['train_loss'][-1]:.4f}")
        print(f"Final train accuracy: {epoch_metrics['train_accuracy'][-1]*100:.2f}%")
        final_val = epoch_metrics['val_accuracy'][-1]
        if final_val == final_val:
            print(f"Final val accuracy: {final_val*100:.2f}%")
        print(f"{'='*70}")

    epoch_metrics['total_training_time_seconds'] = total_time
    epoch_metrics['_optimizer_state_dict'] = optimizer.state_dict()

    return epoch_metrics

In [ ]:
# =============================================================================
# TRAINING RESULTS STORAGE
# =============================================================================

# Store results for all trained models
ALL_RESULTS = {}

print(f"\n{'='*70}")
print(f"TRAINING {len(DISTANCES)} SPECIALIST MODELS")
print(f"{'='*70}")
print(f"Distances to train: {DISTANCES}")
print(f"Each model will be trained on {SAMPLES_PER_MODEL:,} samples")

In [ ]:
# =============================================================================
# MAIN TRAINING LOOP - Train one specialist model per distance
# =============================================================================

import gc

for dist_idx, CURRENT_DISTANCE in enumerate(DISTANCES):
    print(f"\n{'#'*70}")
    print(f"# TRAINING MODEL {dist_idx + 1}/{len(DISTANCES)}: d={CURRENT_DISTANCE} SPECIALIST")
    print(f"{'#'*70}")
    
    EXPERIMENT_NAME = f"d{CURRENT_DISTANCE}"
    
    # -------------------------------------------------------------------------
    # Training config for this distance
    # -------------------------------------------------------------------------
    TRAINING_CONFIG = {
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'samples_per_model': SAMPLES_PER_MODEL,
        'train_samples_per_block': TRAIN_SAMPLES_PER_BLOCK,
        'refresh_every': REFRESH_EVERY,
        'validate_every': VALIDATE_EVERY,
        'p_values': P_VALUES,
        'p_weights': P_WEIGHTS,
        'k_neighbors': K_NEIGHBORS,
        'seed': SEED,
        'train_ratio': TRAIN_RATIO,
        'val_ratio': VAL_RATIO,
        'test_ratio': TEST_RATIO,
    }
    
    # -------------------------------------------------------------------------
    # Check for existing checkpoint
    # -------------------------------------------------------------------------
    checkpoint = load_checkpoint(EXPERIMENT_NAME, MODELS_DIR)
    
    start_epoch = 0
    initial_metrics = None
    optimizer_state_dict = None
    SKIP_TRAINING = False
    
    # Check if we have complete final metrics already saved
    HAS_FINAL_METRICS = False
    if checkpoint is not None:
        if checkpoint.get('training_complete', False):
            print(f"\n{'='*70}")
            print(f"Training already complete for {EXPERIMENT_NAME}")
            print(f"To retrain, delete: {MODELS_DIR / f'{EXPERIMENT_NAME}.pt'}")
            print(f"{'='*70}")
            SKIP_TRAINING = True
            # Check if final_metrics are available (saved after full training + evaluation)
            if 'final_metrics' in checkpoint:
                HAS_FINAL_METRICS = True
                print(f"Final metrics found in checkpoint - will use cached results")
        else:
            start_epoch = checkpoint['current_epoch']
            initial_metrics = checkpoint['epoch_metrics']
            optimizer_state_dict = checkpoint['optimizer_state_dict']
    
    # -------------------------------------------------------------------------
    # Load dataset for this distance (skip if we have final metrics)
    # -------------------------------------------------------------------------
    if HAS_FINAL_METRICS:
        print(f"\nSkipping dataset loading - using cached final metrics")
        all_graphs = None
        val_graphs = None
        test_graphs = None
        dataset_info = checkpoint.get('dataset_info', {f'd{CURRENT_DISTANCE}': {'samples': SAMPLES_PER_MODEL}})
    else:
        all_graphs, dataset_info = load_single_distance_dataset(CURRENT_DISTANCE, SAMPLES_PER_MODEL)
        _, val_graphs, test_graphs = split_dataset(all_graphs, TRAIN_RATIO, VAL_RATIO, SEED)
        del _  # Don't need train split from cache; we refresh training data
    
    # -------------------------------------------------------------------------
    # Initialize model (skip if using cached final metrics)
    # -------------------------------------------------------------------------
    if HAS_FINAL_METRICS:
        print(f"\nSkipping model initialization - using cached results")
        model = None
        total_params = checkpoint.get('model_info', {}).get('total_parameters', 0)
        trainable_params = checkpoint.get('model_info', {}).get('trainable_parameters', 0)
        if total_params == 0:
            # Estimate from config if not saved
            total_params = 50000  # placeholder
            trainable_params = 50000
    else:
        print(f"\n{'='*70}")
        print(f"Initializing GraphSAGE model for d={CURRENT_DISTANCE}")
        print(f"{'='*70}")
        
        model = GraphSAGE(
            nickname=f"specialist_d{CURRENT_DISTANCE}",
            in_channels=IN_CHANNELS,
            hidden_dim=BEST_HYPERPARAMS['hidden_dim'],
            num_layers=BEST_HYPERPARAMS['num_layers'],
            dropout=BEST_HYPERPARAMS['dropout'],
            aggr=BEST_HYPERPARAMS['aggr'],
            device=device,
            base_path=BASE_PATH,
            seed=SEED
        )
        
        # Load model weights from checkpoint (for resuming OR for skipped/complete training)
        if checkpoint is not None:
            model.model.load_state_dict(checkpoint['state_dict'])
            if checkpoint.get('training_complete', False):
                print(f"Loaded trained model weights (training was already complete)")
            else:
                print(f"Loaded model weights from checkpoint (epoch {start_epoch})")
        
        # Count parameters
        total_params = sum(p.numel() for p in model.model.parameters())
        trainable_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)
        print(f"\nModel parameters:")
        print(f"  Total: {total_params:,}")
        print(f"  Trainable: {trainable_params:,}")
    
    # -------------------------------------------------------------------------
    # Train the model
    # -------------------------------------------------------------------------
    if HAS_FINAL_METRICS:
        print(f"\nUsing cached epoch_metrics and final_metrics from checkpoint")
        epoch_metrics = checkpoint['epoch_metrics']
        epoch_metrics['total_training_time_seconds'] = sum(epoch_metrics.get('epoch_time_seconds', [0]))
        # Load final metrics from checkpoint
        final_train_acc = checkpoint['final_metrics']['train_accuracy']
        final_val_acc = checkpoint['final_metrics']['val_accuracy']
        final_test_acc = checkpoint['final_metrics']['test_accuracy']
        print(f"Train accuracy: {final_train_acc*100:.2f}%")
        print(f"Validation accuracy: {final_val_acc*100:.2f}%")
        print(f"Test accuracy: {final_test_acc*100:.2f}%")
    elif SKIP_TRAINING:
        print(f"\nSkipping training - model already complete.")
        print(f"Loading epoch_metrics from checkpoint...")
        epoch_metrics = checkpoint['epoch_metrics']
        epoch_metrics['total_training_time_seconds'] = sum(epoch_metrics.get('epoch_time_seconds', [0]))
        
        # Still need to run final evaluation since no final_metrics in checkpoint
        print(f"\n{'='*70}")
        print(f"Final Evaluation for d={CURRENT_DISTANCE}")
        print(f"{'='*70}")
        
        last_block_idx = (EPOCHS - 1) // REFRESH_EVERY
        train_eval_subset = generate_fresh_training_graphs_single(CURRENT_DISTANCE, n_samples=10_000, block_idx=last_block_idx)
        
        final_train_acc = evaluate_model(model, train_eval_subset, device, BATCH_SIZE)
        final_val_acc = evaluate_model(model, val_graphs, device, BATCH_SIZE)
        final_test_acc = evaluate_model(model, test_graphs, device, BATCH_SIZE)
        
        print(f"Train accuracy (subset): {final_train_acc*100:.2f}%")
        print(f"Validation accuracy: {final_val_acc*100:.2f}%")
        print(f"Test accuracy: {final_test_acc*100:.2f}%")
        
        del train_eval_subset
    else:
        # Training data generator for this distance
        def _make_train_graphs(block_idx: int, d=CURRENT_DISTANCE) -> list:
            return generate_fresh_training_graphs_single(d, TRAIN_SAMPLES_PER_BLOCK, block_idx)
        
        epoch_metrics = train_with_logging(
            model=model,
            val_graphs=val_graphs,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            lr=BEST_HYPERPARAMS['learning_rate'],
            device=device,
            generate_train_graphs_fn=_make_train_graphs,
            refresh_every=REFRESH_EVERY,
            validate_every=VALIDATE_EVERY,
            experiment_name=EXPERIMENT_NAME,
            distances=[CURRENT_DISTANCE],
            hyperparams=BEST_HYPERPARAMS,
            training_config=TRAINING_CONFIG,
            models_dir=MODELS_DIR,
            start_epoch=start_epoch,
            initial_metrics=initial_metrics,
            optimizer_state_dict=optimizer_state_dict,
            verbose=True
        )
        
        # -------------------------------------------------------------------------
        # Final evaluation (only after new training)
        # -------------------------------------------------------------------------
        print(f"\n{'='*70}")
        print(f"Final Evaluation for d={CURRENT_DISTANCE}")
        print(f"{'='*70}")
        
        last_block_idx = (EPOCHS - 1) // REFRESH_EVERY
        train_eval_subset = generate_fresh_training_graphs_single(CURRENT_DISTANCE, n_samples=10_000, block_idx=last_block_idx)
        
        final_train_acc = evaluate_model(model, train_eval_subset, device, BATCH_SIZE)
        final_val_acc = evaluate_model(model, val_graphs, device, BATCH_SIZE)
        final_test_acc = evaluate_model(model, test_graphs, device, BATCH_SIZE)
        
        print(f"Train accuracy (subset): {final_train_acc*100:.2f}%")
        print(f"Validation accuracy: {final_val_acc*100:.2f}%")
        print(f"Test accuracy: {final_test_acc*100:.2f}%")
        
        del train_eval_subset
    
    # -------------------------------------------------------------------------
    # Store results
    # -------------------------------------------------------------------------
    ALL_RESULTS[CURRENT_DISTANCE] = {
        'experiment_name': EXPERIMENT_NAME,
        'model': model,  # May be None if using cached results
        'checkpoint': checkpoint,  # Store checkpoint for saving artifacts
        'epoch_metrics': epoch_metrics,
        'dataset_info': dataset_info,
        'final_train_acc': final_train_acc,
        'final_val_acc': final_val_acc,
        'final_test_acc': final_test_acc,
        'total_params': total_params,
        'trainable_params': trainable_params,
        'val_graphs': val_graphs,
        'test_graphs': test_graphs,
        'has_final_metrics': HAS_FINAL_METRICS,
    }
    
    # Clean up to free memory before next distance
    if all_graphs is not None:
        del all_graphs
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n{'#'*70}")
print(f"# ALL {len(DISTANCES)} SPECIALIST MODELS TRAINED!")
print(f"{'#'*70}")

In [ ]:
# =============================================================================
# SUMMARY OF ALL TRAINED MODELS
# =============================================================================

print(f"\n{'='*70}")
print(f"SUMMARY: All {len(DISTANCES)} Specialist Models")
print(f"{'='*70}")
print(f"\n{'Distance':<10} {'Train Acc':<12} {'Val Acc':<12} {'Test Acc':<12}")
print(f"{'-'*46}")

for d in DISTANCES:
    r = ALL_RESULTS[d]
    print(f"d={d:<7} {r['final_train_acc']*100:>10.2f}% {r['final_val_acc']*100:>10.2f}% {r['final_test_acc']*100:>10.2f}%")

## 7. Save All Artifacts

In [ ]:
# =============================================================================
# SAVE FINAL MODELS (with test metrics) FOR ALL DISTANCES
# =============================================================================

print(f"\n{'='*70}")
print(f"Saving Final Models")
print(f"{'='*70}")

for d in DISTANCES:
    r = ALL_RESULTS[d]
    model_path = MODELS_DIR / f"d{d}.pt"
    
    # Get state_dict and config from model or checkpoint
    if r['model'] is not None:
        state_dict = r['model'].model.state_dict()
        config = r['model']._config
    else:
        # Using cached results - get from checkpoint
        state_dict = r['checkpoint']['state_dict']
        config = r['checkpoint'].get('config', {})
    
    # This save includes final test metrics (computed after training completes)
    # The checkpoint format is preserved for compatibility with resume functionality
    model_save_dict = {
        # Checkpoint-compatible fields
        'state_dict': state_dict,
        'current_epoch': EPOCHS,
        'epoch_metrics': r['epoch_metrics'],
        'training_complete': True,
        'config': config,
        'experiment_name': r['experiment_name'],
        'distances': [d],
        'hyperparams': BEST_HYPERPARAMS,
        # 'training_config': TRAINING_CONFIG,
        'dataset_info': r['dataset_info'],
        'model_info': {
            'total_parameters': r['total_params'],
            'trainable_parameters': r['trainable_params'],
        },
        # Additional final metrics (includes test accuracy)
        'final_metrics': {
            'train_accuracy': r['final_train_acc'],
            'val_accuracy': r['final_val_acc'],
            'test_accuracy': r['final_test_acc'],
            'final_loss': r['epoch_metrics']['train_loss'][-1],
        },
        'timestamp': datetime.now().isoformat(),
    }
    
    torch.save(model_save_dict, model_path)
    print(f"  d={d}: {model_path}")


Saving Final Models


NameError: name 'TRAINING_CONFIG' is not defined

In [ ]:
# =============================================================================
# SAVE TRAINING RESULTS JSON FOR ALL DISTANCES
# =============================================================================

print(f"\n{'='*70}")
print(f"Saving Training Results JSON")
print(f"{'='*70}")

for d in DISTANCES:
    r = ALL_RESULTS[d]
    results_path = RESULTS_DIR / f"d{d}_training.json"
    
    training_results = {
        'experiment_name': r['experiment_name'],
        'distance': d,
        'hyperparams': BEST_HYPERPARAMS,
        'training_config': {
            'epochs': EPOCHS,
            'batch_size': BATCH_SIZE,
            'samples_per_model': SAMPLES_PER_MODEL,
            'p_values': P_VALUES,
            'p_weights': P_WEIGHTS,
            'k_neighbors': K_NEIGHBORS,
            'seed': SEED,
            'train_ratio': TRAIN_RATIO,
            'val_ratio': VAL_RATIO,
            'test_ratio': TEST_RATIO,
        },
        'dataset_info': r['dataset_info'],
        'epoch_metrics': {
            'epochs': r['epoch_metrics']['epoch'],
            'train_loss': r['epoch_metrics']['train_loss'],
            'train_accuracy': r['epoch_metrics']['train_accuracy'],
            'val_accuracy': r['epoch_metrics']['val_accuracy'],
            'epoch_time_seconds': r['epoch_metrics']['epoch_time_seconds'],
            'learning_rate': r['epoch_metrics']['learning_rate'],
            'gpu_memory_mb': r['epoch_metrics']['gpu_memory_mb'],
        },
        'final_metrics': {
            'train_accuracy': r['final_train_acc'],
            'val_accuracy': r['final_val_acc'],
            'test_accuracy': r['final_test_acc'],
            'final_loss': r['epoch_metrics']['train_loss'][-1],
            'total_training_time_seconds': r['epoch_metrics']['total_training_time_seconds'],
        },
        'model_info': {
            'total_parameters': r['total_params'],
            'trainable_parameters': r['trainable_params'],
        },
        'system_info': {
            'device': str(device),
            'gpu_name': GPU_NAME,
            'cuda_version': torch.version.cuda if torch.cuda.is_available() else None,
            'pytorch_version': torch.__version__,
            'python_version': platform.python_version(),
            'platform': platform.platform(),
        },
        'timestamp': datetime.now().isoformat(),
    }
    
    with open(results_path, 'w') as f:
        json.dump(training_results, f, indent=2)
    
    print(f"  d={d}: {results_path}")

In [ ]:
# =============================================================================
# SAVE TRAINING CHARTS FOR ALL DISTANCES
# =============================================================================

print(f"\n{'='*70}")
print(f"Saving Training Charts")
print(f"{'='*70}")

for d in DISTANCES:
    r = ALL_RESULTS[d]
    plot_path = PLOTS_DIR / f"d{d}_training.png"
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs_list = r['epoch_metrics']['epoch']
    
    # Plot 1: Loss curve
    ax1.plot(epochs_list, r['epoch_metrics']['train_loss'], 'b-', linewidth=2, label='Train Loss')
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title(f'Training Loss - d{d} Specialist', fontsize=14)
    ax1.legend(loc='upper right')
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(1, EPOCHS)
    
    # Plot 2: Accuracy curves
    ax2.plot(epochs_list, [acc*100 for acc in r['epoch_metrics']['train_accuracy']], 
             'b-', linewidth=2, label='Train Accuracy')
    ax2.plot(epochs_list, [acc*100 for acc in r['epoch_metrics']['val_accuracy']], 
             'r-', linewidth=2, label='Val Accuracy')
    ax2.axhline(y=r['final_test_acc']*100, color='g', linestyle='--', linewidth=1.5, 
                label=f'Test Accuracy ({r["final_test_acc"]*100:.1f}%)')
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title(f'Training/Validation Accuracy - d{d} Specialist', fontsize=14)
    ax2.legend(loc='lower right')
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(1, EPOCHS)
    ax2.set_ylim(50, 100)
    
    plt.tight_layout()
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f"  d={d}: {plot_path}")
    plt.show()
    plt.close()

In [ ]:
# =============================================================================
# SAVE CSV SUMMARY FOR ALL MODELS
# =============================================================================

print(f"\n{'='*70}")
print(f"Saving CSV Summaries")
print(f"{'='*70}")

# Build combined summary dataframe
summary_rows = []
for d in DISTANCES:
    r = ALL_RESULTS[d]
    summary_rows.append({
        'experiment_name': r['experiment_name'],
        'distance': d,
        'samples': SAMPLES_PER_MODEL,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'learning_rate': BEST_HYPERPARAMS['learning_rate'],
        'hidden_dim': BEST_HYPERPARAMS['hidden_dim'],
        'num_layers': BEST_HYPERPARAMS['num_layers'],
        'aggr': BEST_HYPERPARAMS['aggr'],
        'dropout': BEST_HYPERPARAMS['dropout'],
        'final_loss': r['epoch_metrics']['train_loss'][-1],
        'train_accuracy': r['final_train_acc'],
        'val_accuracy': r['final_val_acc'],
        'test_accuracy': r['final_test_acc'],
        'training_time_minutes': r['epoch_metrics']['total_training_time_seconds'] / 60,
        'total_parameters': r['total_params'],
        'device': str(device),
        'gpu_name': GPU_NAME,
        'timestamp': datetime.now().isoformat(),
    })

summary_df = pd.DataFrame(summary_rows)

# Save combined summary
combined_summary_path = RESULTS_DIR / "all_specialists_summary.csv"
summary_df.to_csv(combined_summary_path, index=False)
print(f"Combined summary: {combined_summary_path}")

# Save individual summaries
for d in DISTANCES:
    individual_path = RESULTS_DIR / f"d{d}_summary.csv"
    summary_df[summary_df['distance'] == d].to_csv(individual_path, index=False)
    print(f"  d={d}: {individual_path}")

# Display summary
print(f"\nAll Specialist Models Summary:")
print(summary_df.to_string(index=False))

## 8. Final Summary

In [ ]:
# =============================================================================
# FINAL SUMMARY
# =============================================================================

print(f"\n{'='*70}")
print(f"TRAINING COMPLETE - {len(DISTANCES)} SPECIALIST MODELS")
print(f"{'='*70}")

print(f"\nConfiguration:")
print(f"  Distances trained: {DISTANCES}")
print(f"  Samples per model: {SAMPLES_PER_MODEL:,}")
print(f"  Epochs per model: {EPOCHS}")

print(f"\nHyperparameters (shared):")
for k, v in BEST_HYPERPARAMS.items():
    print(f"  {k}: {v}")

print(f"\nFinal Results by Distance:")
print(f"{'Distance':<10} {'Train':<10} {'Val':<10} {'Test':<10} {'Time (min)':<12}")
print(f"{'-'*52}")
total_time = 0
for d in DISTANCES:
    r = ALL_RESULTS[d]
    train_time = r['epoch_metrics']['total_training_time_seconds'] / 60
    total_time += train_time
    print(f"d={d:<7} {r['final_train_acc']*100:>8.2f}% {r['final_val_acc']*100:>8.2f}% {r['final_test_acc']*100:>8.2f}% {train_time:>10.1f}")
print(f"{'-'*52}")
print(f"{'Total training time:':<42} {total_time:>10.1f} min")

print(f"\nSaved Artifacts (per distance):")
for d in DISTANCES:
    print(f"  d={d}:")
    print(f"    Model: {MODELS_DIR / f'd{d}.pt'}")
    print(f"    Results: {RESULTS_DIR / f'd{d}_training.json'}")
    print(f"    Chart: {PLOTS_DIR / f'd{d}_training.png'}")
    print(f"    Summary: {RESULTS_DIR / f'd{d}_summary.csv'}")
print(f"\n  Combined summary: {RESULTS_DIR / 'all_specialists_summary.csv'}")

print(f"\n{'='*70}")

In [ ]:
# Display per-epoch metrics table for each model
for d in DISTANCES:
    r = ALL_RESULTS[d]
    print(f"\nPer-Epoch Metrics for d={d}:")
    print(f"{'='*70}")
    
    metrics_df = pd.DataFrame({
        'Epoch': r['epoch_metrics']['epoch'],
        'Train Loss': r['epoch_metrics']['train_loss'],
        'Train Acc': [f"{acc*100:.2f}%" for acc in r['epoch_metrics']['train_accuracy']],
        'Val Acc': [f"{acc*100:.2f}%" for acc in r['epoch_metrics']['val_accuracy']],
        'Time (s)': [f"{t:.1f}" for t in r['epoch_metrics']['epoch_time_seconds']],
        'GPU (MB)': [f"{m:.0f}" for m in r['epoch_metrics']['gpu_memory_mb']],
    })
    
    # Show first 5 and last 5 epochs
    if len(metrics_df) > 10:
        print(metrics_df.head(5).to_string(index=False))
        print("...")
        print(metrics_df.tail(5).to_string(index=False))
    else:
        print(metrics_df.to_string(index=False))

In [ ]:
from google.colab import runtime
runtime.unassign()

## 9. Extract Results from Checkpoints (Recovery Utility)

**Use this cell if training completed but results weren't saved!**

In [4]:
ALL_RESULTS = []

In [8]:
# =============================================================================
# RECOVERY UTILITY: Extract results from checkpoints and save them
# =============================================================================
# Use this if training completed but the saving cells weren't run
# This will load checkpoints and extract all epoch metrics, then save them

print(f"\n{'='*70}")
print(f"RECOVERY: Extracting Results from Checkpoints")
print(f"{'='*70}")

# Get DISTANCES - try to use existing or auto-detect from checkpoints
try:
    _distances = DISTANCES
except NameError:
    # Auto-detect from checkpoint files
    checkpoint_files = list(MODELS_DIR.glob("d*.pt"))
    _distances = [int(f.stem[1:]) for f in checkpoint_files if f.stem.startswith('d') and f.stem[1:].isdigit()]
    _distances.sort()
    print(f"Note: DISTANCES not defined, auto-detected from checkpoints: {_distances}")

# Check if ALL_RESULTS is empty or incomplete
if not ALL_RESULTS or any('epoch_metrics' not in ALL_RESULTS.get(d, {}) for d in _distances):
    print("ALL_RESULTS is empty or incomplete. Loading from checkpoints...")
    
    # Re-initialize ALL_RESULTS from checkpoints
    ALL_RESULTS = {}
    
    for d in _distances:
        checkpoint_path = MODELS_DIR / f"d{d}.pt"
        
        if not checkpoint_path.exists():
            print(f"  ✗ No checkpoint found for d={d}: {checkpoint_path}")
            continue
        
        print(f"\n  Loading checkpoint for d={d}...")
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
        
        # Extract epoch metrics
        epoch_metrics = checkpoint.get('epoch_metrics', {})
        if not epoch_metrics:
            print(f"  ✗ No epoch_metrics in checkpoint for d={d}")
            continue
        
        # Calculate total_training_time_seconds if missing
        if 'total_training_time_seconds' not in epoch_metrics:
            epoch_time_seconds = epoch_metrics.get('epoch_time_seconds', [])
            epoch_metrics['total_training_time_seconds'] = sum(epoch_time_seconds) if epoch_time_seconds else 0.0
            print(f"    Calculated total_training_time: {epoch_metrics['total_training_time_seconds']:.1f}s")
        
        # Extract final metrics
        final_metrics = checkpoint.get('final_metrics', {})
        
        # Get final accuracies from final_metrics or use last epoch values
        train_acc_list = epoch_metrics.get('train_accuracy', [])
        val_acc_list = epoch_metrics.get('val_accuracy', [])
        
        final_train_acc = final_metrics.get('train_accuracy', train_acc_list[-1] if train_acc_list else 0.0)
        final_val_acc = final_metrics.get('val_accuracy', val_acc_list[-1] if val_acc_list else 0.0)
        final_test_acc = final_metrics.get('test_accuracy', 0.0)
        
        # Get samples_per_model from checkpoint or use default
        training_config = checkpoint.get('training_config', {})
        samples = training_config.get('samples_per_model', 1_000_000)  # Default if not found
        
        # Get dataset_info from checkpoint or create default
        dataset_info = checkpoint.get('dataset_info', {})
        if not dataset_info or f'd{d}' not in dataset_info:
            dataset_info = {f'd{d}': {'samples': samples}}
        
        # Reconstruct ALL_RESULTS entry
        ALL_RESULTS[d] = {
            'experiment_name': checkpoint.get('experiment_name', f'd{d}'),
            'model': None,  # Don't load model weights
            'checkpoint': checkpoint,
            'epoch_metrics': epoch_metrics,
            'dataset_info': dataset_info,
            'final_train_acc': final_train_acc,
            'final_val_acc': final_val_acc,
            'final_test_acc': final_test_acc,
            'total_params': checkpoint.get('model_info', {}).get('total_parameters', 0),
            'trainable_params': checkpoint.get('model_info', {}).get('trainable_parameters', 0),
            'val_graphs': None,
            'test_graphs': None,
            'has_final_metrics': 'final_metrics' in checkpoint,
        }
        
        print(f"  ✓ Loaded {len(epoch_metrics.get('epoch', []))} epochs of metrics")
        if final_metrics:
            print(f"    Final accuracies: Train={final_metrics.get('train_accuracy', 0)*100:.2f}%, "
                  f"Val={final_metrics.get('val_accuracy', 0)*100:.2f}%, "
                  f"Test={final_metrics.get('test_accuracy', 0)*100:.2f}%")
    
    print(f"\n✓ Loaded results for {len(ALL_RESULTS)} distances")
    
    # Set DISTANCES if it wasn't defined (for use in saving cells)
    if 'DISTANCES' not in globals():
        DISTANCES = _distances
        print(f"Set DISTANCES = {DISTANCES}")
    
    print(f"Now run the saving cells (cells 23-26) to save the results!")
else:
    print("ALL_RESULTS already populated. Skipping checkpoint extraction.")


RECOVERY: Extracting Results from Checkpoints
ALL_RESULTS is empty or incomplete. Loading from checkpoints...

  Loading checkpoint for d=3...
    Calculated total_training_time: 4438.8s
  ✓ Loaded 50 epochs of metrics

  Loading checkpoint for d=5...
    Calculated total_training_time: 7698.6s
  ✓ Loaded 50 epochs of metrics

  Loading checkpoint for d=7...
    Calculated total_training_time: 12411.6s
  ✓ Loaded 50 epochs of metrics

  Loading checkpoint for d=9...
    Calculated total_training_time: 22321.0s
  ✓ Loaded 50 epochs of metrics

  Loading checkpoint for d=11...
    Calculated total_training_time: 37551.9s
  ✓ Loaded 44 epochs of metrics

✓ Loaded results for 5 distances
Now run the saving cells (cells 23-26) to save the results!
